In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Cell 11.1 - Overview, paths, and analysis definitions
# Purpose:
# Collapse the 1,287,844 variable unitigs from Notebook 10 into unique
# 176-pathogen presence/absence patterns and identify:
# A. unitigs present in >=1 of the 16 high-MIC pathogens and absent from
#    all 160 remaining blaTEM-1-only pathogens;
# B. unitigs absent from all 16 high-MIC pathogens and present in >=1 of
#    the 160 remaining blaTEM-1-only pathogens.
#
# No recurrence threshold is applied.
# No formal MIC association testing is performed.

from pathlib import Path
import gc
import json
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
PROJECT_ROOT = _repo_root()
NOTEBOOK_DIR = PROJECT_ROOT / "03_Notebooks" / "04_Genome_Comparison"
INTERMEDIATE_DIR = PROJECT_ROOT / "04_Intermediate" / "11_Unitig_Patterns"
RESULTS_TABLE_DIR = PROJECT_ROOT / "05_Results" / "Tables"

UNITIG_DIR = PROJECT_ROOT / "04_Intermediate" / "10_Whole_Chromosome_Unitigs"
UNITIG_MATRIX = UNITIG_DIR / "10_variable_unitig_matrix_176xM.npz"
UNITIG_SAMPLES = UNITIG_DIR / "10_unitig_sample_order.csv"
NB10_QC = RESULTS_TABLE_DIR / "10_unitig_representation_final_QC.csv"

PATTERN_WORDS_FILE = INTERMEDIATE_DIR / "11_unique_pattern_words.npz"
UNITIG_TO_PATTERN_FILE = INTERMEDIATE_DIR / "11_unitig_to_pattern.npz"
PATTERN_SAMPLE_ORDER = INTERMEDIATE_DIR / "11_pattern_sample_order.csv"
PATTERN_METADATA = INTERMEDIATE_DIR / "11_unique_pattern_metadata.csv.gz"
A_UNITIG_FILE = INTERMEDIATE_DIR / "11_A_high_MIC_exclusive_unitigs.csv.gz"
B_UNITIG_FILE = INTERMEDIATE_DIR / "11_B_remaining_pathogen_exclusive_unitigs.csv.gz"
A_PATTERN_FILE = INTERMEDIATE_DIR / "11_A_high_MIC_exclusive_patterns.csv.gz"
B_PATTERN_FILE = INTERMEDIATE_DIR / "11_B_remaining_pathogen_exclusive_patterns.csv.gz"

RECURRENCE_SUMMARY = RESULTS_TABLE_DIR / "11_exclusive_unitig_recurrence_summary.csv"
PATTERN_SUMMARY = RESULTS_TABLE_DIR / "11_unitig_pattern_consolidation_summary.csv"
FINAL_QC_FILE = RESULTS_TABLE_DIR / "11_unitig_pattern_final_QC.csv"
COMPLETION_FILE = INTERMEDIATE_DIR / "11_UNITIG_PATTERN_ANALYSIS_COMPLETE.json"

N_EXPECTED = 176
HIGH_MIC_LOG2_THRESHOLD = 2.0

for path in [PROJECT_ROOT, NOTEBOOK_DIR, RESULTS_TABLE_DIR]:
    assert path.exists(), f"Required path not found: {path}"

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook 11 - Unitig Pattern Consolidation and High-MIC Exclusivity")
print("Notebook folder:", NOTEBOOK_DIR)
print("Output folder:", INTERMEDIATE_DIR)
print("No recurrence threshold is applied.")
print("No formal MIC association testing is performed.")
print("\nTransition: Cell 11.2 will verify Notebook 10 outputs and define the 16/160 groups.")


In [ ]:
#@title Cell 11.2 - Verify Notebook 10 outputs and define the 16/160 groups
# Purpose:
# Verify the accepted Notebook 10 representation and establish the exact
# 16 high-MIC and 160 remaining blaTEM-1-only pathogens.

for path in [UNITIG_MATRIX, UNITIG_SAMPLES, NB10_QC]:
    assert path.exists(), f"Required Notebook 10 input missing: {path}"

nb10_qc = pd.read_csv(NB10_QC)
assert len(nb10_qc) == 1
assert bool(nb10_qc.loc[0, "final_QC_pass"])

samples = pd.read_csv(UNITIG_SAMPLES)

required = {"sample_index", "biosample", "assembly_accession", "log2_mic"}
missing = required - set(samples.columns)
assert not missing, f"Sample-order file missing: {sorted(missing)}"

assert len(samples) == N_EXPECTED
assert samples["biosample"].nunique() == N_EXPECTED

samples = samples.sort_values("sample_index").reset_index(drop=True)
assert np.array_equal(samples["sample_index"].to_numpy(), np.arange(N_EXPECTED))

samples["high_MIC_group"] = samples["log2_mic"] > HIGH_MIC_LOG2_THRESHOLD

n_high = int(samples["high_MIC_group"].sum())
n_remaining = int((~samples["high_MIC_group"]).sum())

assert n_high == 16, f"Expected 16 high-MIC pathogens, found {n_high}."
assert n_remaining == 160, f"Expected 160 remaining pathogens, found {n_remaining}."

high_indices = samples.loc[samples["high_MIC_group"], "sample_index"].astype(int).to_numpy()
remaining_indices = samples.loc[~samples["high_MIC_group"], "sample_index"].astype(int).to_numpy()

samples.to_csv(PATTERN_SAMPLE_ORDER, index=False)

print("Verified Notebook 10 final QC: PASS")
print("Total pathogens:", len(samples))
print("16 high-MIC pathogens:", n_high)
print("160 remaining pathogens:", n_remaining)
print("High-MIC definition: log2 MIC > 2, matching the established 16-pathogen group.")

display(
    samples.loc[
        samples["high_MIC_group"],
        ["sample_index", "biosample", "assembly_accession", "log2_mic"],
    ]
)

print("\nCell 11.2 complete.")
print("Transition: Cell 11.3 will collapse unitigs into unique presence/absence patterns.")


In [ ]:
#@title Cell 11.3 - Collapse unitigs into unique 176-pathogen presence/absence patterns
# Purpose:
# Encode each unitig pattern in three 64-bit words and collapse identical
# patterns. Unitigs sharing one pattern have exactly the same 0/1 values
# across all 176 pathogens.

print("Loading Notebook 10 sparse matrix...")

unitig_matrix = sparse.load_npz(UNITIG_MATRIX).tocsc()
assert unitig_matrix.shape[0] == N_EXPECTED

M = int(unitig_matrix.shape[1])
expected_M = int(nb10_qc.loc[0, "variable_unitigs_M"])
assert M == expected_M

print("Unitigs M:", f"{M:,}")
print("Matrix shape:", unitig_matrix.shape)
print("Nonzero entries:", f"{unitig_matrix.nnz:,}")

matrix_csr = unitig_matrix.tocsr()
del unitig_matrix
gc.collect()

packed_patterns = np.zeros((M, 3), dtype=np.uint64)

print("\nPacking 176 rows into three uint64 words per unitig...")

for row_index in range(N_EXPECTED):
    start = matrix_csr.indptr[row_index]
    end = matrix_csr.indptr[row_index + 1]
    cols = matrix_csr.indices[start:end]

    word_index = row_index // 64
    bit_index = row_index % 64
    bit_mask = np.uint64(1) << np.uint64(bit_index)

    packed_patterns[cols, word_index] |= bit_mask

del matrix_csr
gc.collect()

print("Packed array:", packed_patterns.shape)
print("Packed size:", f"{packed_patterns.nbytes / 1024**2:.1f} MiB")
print("\nCollapsing identical patterns...")

(
    unique_words,
    first_unitig_index,
    unitig_to_pattern,
    unitigs_per_pattern,
) = np.unique(
    packed_patterns,
    axis=0,
    return_index=True,
    return_inverse=True,
    return_counts=True,
)

del packed_patterns
gc.collect()

unitig_to_pattern = unitig_to_pattern.astype(np.int32, copy=False)
first_unitig_index = first_unitig_index.astype(np.int32, copy=False)
unitigs_per_pattern = unitigs_per_pattern.astype(np.int32, copy=False)

P = int(len(unitigs_per_pattern))

assert int(unitigs_per_pattern.sum()) == M
assert unitig_to_pattern.shape == (M,)
assert unique_words.shape == (P, 3)

np.savez_compressed(PATTERN_WORDS_FILE, unique_words=unique_words)
np.savez_compressed(UNITIG_TO_PATTERN_FILE, unitig_to_pattern=unitig_to_pattern)

print("\nUnique presence/absence patterns P:", f"{P:,}")
print("Original unitigs M:", f"{M:,}")
print("Average unitigs per pattern:", f"{M / P:.2f}")
print("Saved packed pattern representation.")
print("\nCell 11.3 complete.")
print("Transition: Cell 11.4 will classify A, B, and shared patterns.")


In [ ]:
#@title Cell 11.4 - Classify A/B exclusive patterns and count recurrence
# Purpose:
# Classify each unique pattern as:
# A. high-MIC-exclusive;
# B. remaining-pathogen-exclusive; or
# C. present in both groups.
#
# No recurrence cutoff is imposed.

def count_presence(words, row_indices):
    counts = np.zeros(words.shape[0], dtype=np.int16)

    for row_index in row_indices:
        word_index = int(row_index) // 64
        bit_index = int(row_index) % 64

        counts += (
            (words[:, word_index] >> np.uint64(bit_index))
            & np.uint64(1)
        ).astype(np.int16)

    return counts

high_count = count_presence(unique_words, high_indices)
remaining_count = count_presence(unique_words, remaining_indices)
present_count = high_count + remaining_count

assert present_count.min() >= 1
assert present_count.max() <= 175

is_A = (high_count >= 1) & (remaining_count == 0)
is_B = (high_count == 0) & (remaining_count >= 1)
is_shared = (high_count >= 1) & (remaining_count >= 1)

assert np.all(is_A | is_B | is_shared)
assert not np.any(is_A & is_B)

category_code = np.zeros(P, dtype=np.uint8)
category_code[is_A] = 1
category_code[is_B] = 2

labels = np.array(
    [
        "shared_between_groups",
        "A_high_MIC_exclusive",
        "B_remaining_pathogen_exclusive",
    ],
    dtype=object,
)

pattern_metadata = pd.DataFrame(
    {
        "pattern_id": np.arange(P, dtype=np.int32),
        "first_unitig_index": first_unitig_index,
        "n_unitigs": unitigs_per_pattern,
        "present_count": present_count,
        "high_MIC_count": high_count,
        "remaining_count": remaining_count,
        "category": labels[category_code],
    }
)

pattern_metadata.to_csv(PATTERN_METADATA, index=False, compression="gzip")
pattern_metadata.loc[is_A].to_csv(A_PATTERN_FILE, index=False, compression="gzip")
pattern_metadata.loc[is_B].to_csv(B_PATTERN_FILE, index=False, compression="gzip")

n_A_patterns = int(is_A.sum())
n_B_patterns = int(is_B.sum())
n_shared_patterns = int(is_shared.sum())

n_A_unitigs = int(unitigs_per_pattern[is_A].sum())
n_B_unitigs = int(unitigs_per_pattern[is_B].sum())
n_shared_unitigs = int(unitigs_per_pattern[is_shared].sum())

print("Unique patterns P:", f"{P:,}")
print("A. High-MIC-exclusive patterns:", f"{n_A_patterns:,}")
print("A. High-MIC-exclusive unitigs:", f"{n_A_unitigs:,}")
print("B. Remaining-pathogen-exclusive patterns:", f"{n_B_patterns:,}")
print("B. Remaining-pathogen-exclusive unitigs:", f"{n_B_unitigs:,}")
print("Patterns present in both groups:", f"{n_shared_patterns:,}")
print("Unitigs present in both groups:", f"{n_shared_unitigs:,}")

rows = []

for recurrence in range(1, 17):
    mask = is_A & (high_count == recurrence)
    rows.append(
        {
            "category": "A_high_MIC_exclusive",
            "recurrence_within_relevant_group": recurrence,
            "relevant_group_size": 16,
            "n_unique_patterns": int(mask.sum()),
            "n_unitigs": int(unitigs_per_pattern[mask].sum()),
        }
    )

for recurrence in range(1, 161):
    mask = is_B & (remaining_count == recurrence)
    rows.append(
        {
            "category": "B_remaining_pathogen_exclusive",
            "recurrence_within_relevant_group": recurrence,
            "relevant_group_size": 160,
            "n_unique_patterns": int(mask.sum()),
            "n_unitigs": int(unitigs_per_pattern[mask].sum()),
        }
    )

recurrence_summary = pd.DataFrame(rows)
recurrence_summary.to_csv(RECURRENCE_SUMMARY, index=False)

print("\nA recurrence across the 16 high-MIC pathogens:")
display(
    recurrence_summary.loc[
        (recurrence_summary["category"] == "A_high_MIC_exclusive")
        & (recurrence_summary["n_unitigs"] > 0)
    ]
)

print("\nB recurrence across the 160 remaining pathogens:")
display(
    recurrence_summary.loc[
        (recurrence_summary["category"] == "B_remaining_pathogen_exclusive")
        & (recurrence_summary["n_unitigs"] > 0)
    ]
)

print("\nCell 11.4 complete.")
print("Transition: Cell 11.5 will save direct A/B unitig lookup tables and the summary.")


In [ ]:
#@title Cell 11.5 - Save A/B unitig lookup tables and consolidation summary
# Purpose:
# Save direct unitig-level lookup tables for A and B and the compact
# pattern-consolidation summary.

unitig_category_code = category_code[unitig_to_pattern]

A_unitig_indices = np.flatnonzero(unitig_category_code == 1).astype(np.int32)
B_unitig_indices = np.flatnonzero(unitig_category_code == 2).astype(np.int32)

assert len(A_unitig_indices) == n_A_unitigs
assert len(B_unitig_indices) == n_B_unitigs

def build_lookup(unitig_indices, group):
    pattern_ids = unitig_to_pattern[unitig_indices]

    if group == "A":
        recurrence = high_count[pattern_ids]
    else:
        recurrence = remaining_count[pattern_ids]

    out = pd.DataFrame(
        {
            "unitig_index": unitig_indices,
            "pattern_id": pattern_ids,
            "recurrence_within_relevant_group": recurrence,
            "n_unitigs_sharing_pattern": unitigs_per_pattern[pattern_ids],
        }
    )

    out.insert(
        1,
        "unitig_id",
        [f"U{int(i) + 1:09d}" for i in unitig_indices],
    )

    return out

A_lookup = build_lookup(A_unitig_indices, "A")
B_lookup = build_lookup(B_unitig_indices, "B")

A_lookup.to_csv(A_UNITIG_FILE, index=False, compression="gzip")
B_lookup.to_csv(B_UNITIG_FILE, index=False, compression="gzip")

summary_row = {
    "n_pathogens": N_EXPECTED,
    "n_high_MIC_pathogens": 16,
    "n_remaining_pathogens": 160,
    "original_variable_unitigs_M": M,
    "unique_presence_absence_patterns_P": P,
    "pattern_reduction_factor_M_over_P": M / P,
    "A_high_MIC_exclusive_patterns": n_A_patterns,
    "A_high_MIC_exclusive_unitigs": n_A_unitigs,
    "B_remaining_pathogen_exclusive_patterns": n_B_patterns,
    "B_remaining_pathogen_exclusive_unitigs": n_B_unitigs,
    "shared_between_groups_patterns": n_shared_patterns,
    "shared_between_groups_unitigs": n_shared_unitigs,
    "recurrence_threshold_applied": False,
    "formal_MIC_association_performed": False,
}

pd.DataFrame([summary_row]).to_csv(PATTERN_SUMMARY, index=False)

print("Saved A unitig lookup:", A_UNITIG_FILE)
print("Saved B unitig lookup:", B_UNITIG_FILE)
print("Saved pattern metadata:", PATTERN_METADATA)
print("Saved consolidation summary:", PATTERN_SUMMARY)

print("\nOutput sizes:")
for path in [
    PATTERN_WORDS_FILE,
    UNITIG_TO_PATTERN_FILE,
    PATTERN_METADATA,
    A_UNITIG_FILE,
    B_UNITIG_FILE,
]:
    print("-", path.name, f"{path.stat().st_size / 1024**2:.1f} MiB")

print("\nCell 11.5 complete.")
print("Transition: Cell 11.6 will independently verify the consolidation and A/B definitions.")


In [ ]:
#@title Cell 11.6 - Independent QC of pattern consolidation and A/B definitions
# Purpose:
# Verify that every original unitig maps to exactly one unique pattern,
# A/B definitions are exact, and selected original unitig columns match
# their stored packed patterns.

loaded_words = np.load(PATTERN_WORDS_FILE)["unique_words"]
loaded_mapping = np.load(UNITIG_TO_PATTERN_FILE)["unitig_to_pattern"]
loaded_meta = pd.read_csv(PATTERN_METADATA)

assert loaded_words.shape == (P, 3)
assert loaded_mapping.shape == (M,)
assert len(loaded_meta) == P

mapping_counts = np.bincount(loaded_mapping, minlength=P).astype(np.int32)

assert np.array_equal(mapping_counts, unitigs_per_pattern)
assert int(mapping_counts.sum()) == M

A_meta = loaded_meta.loc[loaded_meta["category"] == "A_high_MIC_exclusive"]
B_meta = loaded_meta.loc[loaded_meta["category"] == "B_remaining_pathogen_exclusive"]

assert ((A_meta["high_MIC_count"] >= 1) & (A_meta["remaining_count"] == 0)).all()
assert ((B_meta["high_MIC_count"] == 0) & (B_meta["remaining_count"] >= 1)).all()

check_matrix = sparse.load_npz(UNITIG_MATRIX).tocsc()

rng = np.random.default_rng(20260913)
n_random = min(200, M)

random_unitigs = rng.choice(M, size=n_random, replace=False)
check_unitigs = np.unique(
    np.concatenate(
        [
            np.array([0, M - 1], dtype=np.int64),
            random_unitigs.astype(np.int64),
        ]
    )
)

mismatches = 0

for unitig_index in check_unitigs:
    pattern_id = int(loaded_mapping[unitig_index])
    words = loaded_words[pattern_id]

    expected_rows = set()

    for row_index in range(N_EXPECTED):
        word_index = row_index // 64
        bit_index = row_index % 64

        present = int(
            (words[word_index] >> np.uint64(bit_index))
            & np.uint64(1)
        )

        if present:
            expected_rows.add(row_index)

    start = check_matrix.indptr[unitig_index]
    end = check_matrix.indptr[unitig_index + 1]
    observed_rows = set(check_matrix.indices[start:end].tolist())

    if expected_rows != observed_rows:
        mismatches += 1

del check_matrix
gc.collect()

assert mismatches == 0
assert n_A_unitigs + n_B_unitigs + n_shared_unitigs == M
assert n_A_patterns + n_B_patterns + n_shared_patterns == P

qc_row = {
    "original_unitigs_M": M,
    "unique_patterns_P": P,
    "all_unitigs_mapped_once": True,
    "pattern_counts_sum_to_M": True,
    "A_definition_exact": True,
    "B_definition_exact": True,
    "sampled_unitig_pattern_matches": int(len(check_unitigs)),
    "sampled_unitig_pattern_mismatches": mismatches,
    "category_pattern_counts_sum_to_P": True,
    "category_unitig_counts_sum_to_M": True,
    "final_QC_pass": True,
}

pd.DataFrame([qc_row]).to_csv(FINAL_QC_FILE, index=False)

completion = {
    "status": "complete",
    "original_unitigs_M": int(M),
    "unique_patterns_P": int(P),
    "A_high_MIC_exclusive_unitigs": int(n_A_unitigs),
    "B_remaining_pathogen_exclusive_unitigs": int(n_B_unitigs),
    "final_QC_pass": True,
}

COMPLETION_FILE.write_text(
    json.dumps(completion, indent=2),
    encoding="utf-8",
)

print("Original unitigs M:", f"{M:,}")
print("Unique patterns P:", f"{P:,}")
print("All unitigs mapped exactly once: PASS")
print("A definition: PASS")
print("B definition: PASS")
print("Direct pattern checks:", len(check_unitigs), "unitigs,", mismatches, "mismatches")
print("Final QC: PASS")
print("Saved final QC:", FINAL_QC_FILE)

print("\nCell 11.6 complete.")
print("Transition: Cell 11.7 will summarize the results and close Notebook 11.")


In [ ]:
#@title Cell 11.7 - Final summary and stopping point
# Purpose:
# Summarize Notebook 11 and stop before formal MIC association testing.

summary = pd.read_csv(PATTERN_SUMMARY)
qc = pd.read_csv(FINAL_QC_FILE)
recurrence = pd.read_csv(RECURRENCE_SUMMARY)

display(summary.T.rename(columns={0: "value"}))

print("\nA. High-MIC-exclusive recurrence:")
display(
    recurrence.loc[
        (recurrence["category"] == "A_high_MIC_exclusive")
        & (recurrence["n_unitigs"] > 0)
    ]
)

print("\nB. Remaining-pathogen-exclusive recurrence:")
display(
    recurrence.loc[
        (recurrence["category"] == "B_remaining_pathogen_exclusive")
        & (recurrence["n_unitigs"] > 0)
    ]
)

print("\nFinal QC:", bool(qc.loc[0, "final_QC_pass"]))
assert bool(qc.loc[0, "final_QC_pass"])

print("\nFINAL STATUS: unique unitig patterns and A/B exclusive unitig sets accepted.")
print("Notebook 11 ends here.")
print("No recurrence cutoff has been applied.")
print("Do not begin formal continuous-MIC association testing until these results have been reviewed.")
